In [ ]:
from qulacs import QuantumCircuit as QulacsCircuit
from qulacs.gate import DenseMatrix, CNOT, RX, RZ, to_matrix_gate
from qulacs import QuantumState
import numpy as np
import math

from sub_function_1D import *

### Parameter Settings

In [ ]:
# Time step size
tau =1
# Number of time steps
Total_steps = 200
# Total time
T = Total_steps*tau
# Number of spatial grid points
num_grid = 8
# Spatial step size
delta_x = 1
# Scale normalization coefficient
Lambda = 10**4  # Factor to reduce the nonlinear term, 1 might also work
# Density
density = 1
# Permittivity
epsilon_0 = 1
# Mass
mass = 1
# Charge
q = -1
# Number of variables
N = 2*num_grid
# Upper limit of total particle number
m = 1
# Size of Hamiltonian matrix
M = math.comb(m+N, m)
# Number of x qubits
n_x = math.floor(np.log2(M))+1
# Number of ancilla qubits required for U
n_a = 1
eta=1
T_real=Total_steps*tau/(192*eta*(m/2)**(5/2))

k = 2*np.pi/num_grid
x_min = -num_grid/2
x_max = num_grid/2

### Initial Variable Preparation
$
\bold{x} = \begin{bmatrix}
            u(0,0) \\
            u(\Delta x,0) \\
            u(2\Delta x,0) \\
            \vdots \\
            E(0,0) \\
            E(\Delta x,0) \\
            E(2\Delta x,0) \\
            \vdots 
\end{bmatrix}
$

In [ ]:
r = np.linspace(x_min, x_max, num_grid)
# u list
u = np.zeros((num_grid,Total_steps+1))
u[:,0] = np.ones(num_grid)
# E list
E = np.zeros((num_grid,Total_steps+1))
# Variable list
x = np.concatenate((u[:,0], E[:,0]))
normalize_matrix(x)

### Initial State Preparation
$
|\psi(\bold{x},0)\rangle = \begin{bmatrix}
                    \psi_{m=0} \\
                    cu(0,0) \\
                    cu(\Delta x,0) \\
                    cu(2\Delta x,0) \\
                    \vdots \\
                    cE(0,0) \\
                    cE(\Delta x,0) \\
                    cE(2\Delta x,0) \\
                    \vdots \\
                    \psi_{m=2}
\end{bmatrix},
\|\psi(\bold{x},0)\| = constant
$

In [ ]:
state_preparation(2*num_grid,M,x,Lambda)

### KvN-expm Hamiltonian Simulation Time Evolution
Sequential time evolution with small time step $\tau$  
$|\psi(\bold{x},t+1) \rangle = \exp(-i\frac{H}{\alpha}\tau)|\psi(\bold{x},t) \rangle$

In [ ]:
u,E,alpha = HS_TestSim_U_Hamiltonian_matrix(n_x, n_a, tau, delta_x, Lambda, density, epsilon_0, mass, q, m, num_grid, M, u, E, Total_steps)

In [ ]:
# Save in binary format
filename = 'output/CaseA/1DPlasmaOscillationTest_u_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(n_x, delta_x, T, tau, m)
np.save(filename, u)
filename = 'output/CaseA/1DPlasmaOscillationTest_E_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(n_x, delta_x, T, tau, m)
np.save(filename, E)